<a href="https://colab.research.google.com/github/Mohamedashraf2005/Flight_Delay-PySpark/blob/main/Flight_Delay_Chain_Reaction_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Flight Delay Chain Reaction Prediction
- A predictive model that determines if a delayed flight will cause delays in following flights, based on timing, routing, and historical delay patterns.

##### Dataset
> https://www.kaggle.com/datasets/yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018/data

##### Env Setup

In [1]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets list

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [2]:
#!kaggle datasets download -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018
!kaggle datasets files -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018

name           size  creationDate                
--------  ---------  --------------------------  
2009.csv  792609318  2019-10-30 02:23:34.345000  
2010.csv  794152224  2019-10-30 02:23:20.703000  
2011.csv  746858940  2019-10-30 02:23:14.136000  
2012.csv  775286574  2019-10-30 02:23:21.062000  
2013.csv  787473333  2019-10-30 02:23:31.079000  
2014.csv  719599833  2019-10-30 02:24:02.425000  
2015.csv  719068948  2019-10-30 02:24:06.559000  
2016.csv  694595243  2019-10-30 02:24:10.506000  
2017.csv  702257855  2019-10-30 02:24:01.970000  
2018.csv  892988892  2019-10-30 02:24:05.651000  


In [3]:
# !kaggle datasets download -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018 -f 2017.csv
!kaggle datasets download -d yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018 -f 2018.csv

Dataset URL: https://www.kaggle.com/datasets/yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018
License(s): other
100% 238M/238M [00:02<00:00, 90.2MB/s]



> /content/airline-delay-and-cancellation-data-2009-2018.zip

In [4]:
# !unzip /content/2017.csv.zip
!unzip /content/2018.csv.zip


Archive:  /content/2018.csv.zip
  inflating: 2018.csv                


#### DataSet OverView[Kaggle]
- **FL_DATE**: Flight operation date (YYYY-MM-DD format).
- **OP_CARRIER**: Two-letter IATA code of the operating airline (e.g., AA, DL).
- **OP_CARRIER_FL_NUM**: Flight number assigned by the operating carrier.
- **ORIGIN**: Three-letter IATA airport code of departure airport.
- **DEST**: Three-letter IATA airport code of destination airport.
- **CRS_DEP_TIME**: Scheduled departure time in local HHMM format (e.g., 1430 = 2:30 PM).
- **DEP_TIME**: Actual departure time in local HHMM format (may be null if cancelled).
- **DEP_DELAY**: Departure delay in minutes (positive = late, negative = early).
- **TAXI_OUT**: Time in minutes from gate pushback to wheels-off (takeoff).
- **WHEELS_OFF**: Timestamp (local HHMM) when aircraft wheels left the ground.
- **WHEELS_ON**: Timestamp (local HHMM) when aircraft wheels touched down at destination.
- **TAXI_IN**: Time in minutes from wheels-on to gate arrival at destination.
- **CRS_ARR_TIME**: Scheduled arrival time in local HHMM format.
- **ARR_TIME**: Actual arrival time in local HHMM format (may be null if cancelled/diverted).
- **ARR_DELAY**: Arrival delay in minutes (positive = late, negative = early).
- **CANCELLED**: Binary flag (1.0 = cancelled, 0.0 = operated).
- **CANCELLATION_CODE**: Reason code for cancellation (A=Carrier, B=Weather, C=NAS, D=Security).
- **DIVERTED**: Binary flag (1.0 = flight diverted to alternate airport, 0.0 = normal).
- **CRS_ELAPSED_TIME**: Scheduled block time (gate-to-gate) in minutes.
- **ACTUAL_ELAPSED_TIME**: Actual block time (gate-to-gate) in minutes.
- **AIR_TIME**: Actual airborne time (wheels-off to wheels-on) in minutes.
- **DISTANCE**: Great-circle distance between origin and destination airports in miles.
- **CARRIER_DELAY**: Portion of total delay attributed to airline-controlled factors (minutes).
- **WEATHER_DELAY**: Portion of delay caused by extreme weather conditions (minutes).
- **NAS_DELAY**: Delay due to National Aviation System constraints (e.g., congestion, staffing) in minutes.
- **SECURITY_DELAY**: Delay caused by security incidents or breaches (minutes).
- **LATE_AIRCRAFT_DELAY**: Delay propagated from previous flight(s) of the same aircraft (minutes).

### PySpark

In [5]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql import Window
import pyspark.sql.functions as F

spark=SparkSession.builder.getOrCreate()

Define the Schema Manually
By default, Spark might scan the whole 7M rows just to figure out which columns are integers and which are strings.

In [6]:
flight_schema = StructType([
    StructField("FL_DATE", StringType(), True),
    StructField("OP_CARRIER", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True),
    StructField("DEP_TIME", DoubleType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("WHEELS_OFF", DoubleType(), True),
    StructField("WHEELS_ON", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", DoubleType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("CARRIER_DELAY", DoubleType(), True),
    StructField("WEATHER_DELAY", DoubleType(), True),
    StructField("NAS_DELAY", DoubleType(), True),
    StructField("SECURITY_DELAY", DoubleType(), True),
    StructField("LATE_AIRCRAFT_DELAY", DoubleType(), True)
])

In [7]:
df = spark.read.csv("2018.csv", header=True, schema=flight_schema)


In [8]:
df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: do

In [9]:
row_count = df.count()

col_count = len(df.columns)

print(f"Shape: ({row_count}, {col_count})")

Shape: (7213446, 27)


In [10]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [11]:
df.limit(5).toPandas()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2018-01-01,UA,2429,EWR,DEN,1517,1512.0,-5.0,15.0,1527.0,1712.0,10.0,1745,1722.0,-23.0,0.0,None,0.0,268.0,250.0,225.0,1605.0,NaN,NaN,NaN,NaN,NaN
1,2018-01-01,UA,2427,LAS,SFO,1115,1107.0,-8.0,11.0,1118.0,1223.0,7.0,1254,1230.0,-24.0,0.0,None,0.0,99.0,83.0,65.0,414.0,NaN,NaN,NaN,NaN,NaN
2,2018-01-01,UA,2426,SNA,DEN,1335,1330.0,-5.0,15.0,1345.0,1631.0,5.0,1649,1636.0,-13.0,0.0,None,0.0,134.0,126.0,106.0,846.0,NaN,NaN,NaN,NaN,NaN
3,2018-01-01,UA,2425,RSW,ORD,1546,1552.0,6.0,19.0,1611.0,1748.0,6.0,1756,1754.0,-2.0,0.0,None,0.0,190.0,182.0,157.0,1120.0,NaN,NaN,NaN,NaN,NaN
4,2018-01-01,UA,2424,ORD,ALB,630,650.0,20.0,13.0,703.0,926.0,10.0,922,936.0,14.0,0.0,None,0.0,112.0,106.0,83.0,723.0,NaN,NaN,NaN,NaN,NaN


In [12]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|      0|         0|                0|     0| 

In [13]:
df.sample(0.05).describe().show()

+-------+----------+----------+------------------+------+------+------------------+------------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+--------------------+-----------------+--------------------+------------------+-------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+-------------------+
|summary|   FL_DATE|OP_CARRIER| OP_CARRIER_FL_NUM|ORIGIN|  DEST|      CRS_DEP_TIME|          DEP_TIME|        DEP_DELAY|         TAXI_OUT|        WHEELS_OFF|         WHEELS_ON|          TAXI_IN|      CRS_ARR_TIME|         ARR_TIME|        ARR_DELAY|           CANCELLED|CANCELLATION_CODE|            DIVERTED|  CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|          AIR_TIME|         DISTANCE|     CARRIER_DELAY|     WEATHER_DELAY|         NAS_DELAY|     SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+-------+----------+----------+---

In [14]:
df.limit(1).toPandas()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2018-01-01,UA,2429,EWR,DEN,1517,1512.0,-5.0,15.0,1527.0,1712.0,10.0,1745,1722.0,-23.0,0.0,None,0.0,268.0,250.0,225.0,1605.0,NaN,NaN,NaN,NaN,NaN


In [15]:
df= df.filter(df["DIVERTED"] == 0)
df= df.filter(df["CANCELLED"] == 0)

df = df.dropna(subset=["ARR_DELAY", "DEP_DELAY"])

delay_cause_cols = ["CARRIER_DELAY", "WEATHER_DELAY",
                    "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]

df = df.fillna(0, subset=delay_cause_cols)

df = df.drop("CANCELLATION_CODE")

##### Handle remaining nulls pleassse ;)

In [16]:
def hhmm_to_minutes(col_name):
    return ((F.col(col_name) / 100).cast(IntegerType()) * 60 +
             F.col(col_name) % 100)

#### Chain Delay Extraction
**Objective:** Create a binary target (`CHAIN_DELAY`) to predict if a flight's delay cascades from a previous flight operated by the same aircraft.

**How it works:**
- **Sequence Alignment:** Used a PySpark `Window` to order flights chronologically per aircraft/route.
- **Historical Lookup:** Applied `lag("ARR_DELAY")` to pull the previous flight's actual arrival delay.
- **Buffer Calculation:** Computed `turnaround_time` as the scheduled gap between the previous arrival and current departure.
- **Cascade Logic:** `CHAIN_DELAY = 1` only when **all three** conditions align:
  - Incoming flight was late (`>15 min`)
  - Ground turnaround is tight (`<90 min`)
  - Current flight also departs late (`>15 min`)


OP_CARRIER + OP_CARRIER_FL_NUM + FL_DATE

In [17]:
window = Window.partitionBy("ORIGIN", "FL_DATE") \
               .orderBy("CRS_DEP_TIME")


In [18]:
df = df.withColumn("prev_crs_arr_minutes",
                   F.lag(hhmm_to_minutes("CRS_ARR_TIME"), 1).over(window))

df = df.withColumn("crs_dep_minutes",
                   hhmm_to_minutes("CRS_DEP_TIME"))


df = df.withColumn("turnaround_time", F.col("crs_dep_minutes") - F.lag(hhmm_to_minutes("CRS_ARR_TIME"), 1).over(window))

# Previous flight's actual arrival delay
df = df.withColumn("prev_arr_delay", F.lag("ARR_DELAY", 1).over(window))

# Current flight's departure delay (what we want to predict)
df = df.withColumn("next_dep_delay",F.lead("DEP_DELAY", 1).over(window))


df = df.withColumn(
    "CHAIN_DELAY",
    (
        (F.col("prev_arr_delay") > 15) &          # previous was late
        (F.col("turnaround_time") < 90) &          # tight turnaround
        (F.col("DEP_DELAY") > 15)                  # current flight IS delayed
    ).cast("int")
)


In [19]:
df.limit(1).toPandas()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,prev_crs_arr_minutes,crs_dep_minutes,turnaround_time,prev_arr_delay,next_dep_delay,CHAIN_DELAY
0,2018-01-04,OO,7427,ABE,DTW,545,543.0,-2.0,14.0,557.0,715.0,17.0,744,732.0,-12.0,0.0,0.0,119.0,109.0,78.0,425.0,0.0,0.0,0.0,0.0,0.0,NaN,345,NaN,NaN,1.0,0


In [20]:
row_count_cleaned_bfdb = df.count()

col_count_cleaned_bddb = len(df.columns)

print(f"Shape: ({row_count_cleaned_bfdb}, {col_count_cleaned_bddb})")

Shape: (7071818, 32)


In [21]:
# prev_arr_delay >>> first flight of the day for each carrier+route. No previous flight exists so lag returns null.
# next_dep_delay>>>last flight of the day, no next flight exists so lead returns null.
df = df.dropna(subset=["prev_arr_delay","next_dep_delay"])

In [22]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+--------------------+---------------+---------------+--------------+--------------+-----------+
|FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|prev_crs_arr_minutes|crs_dep_minutes|turnaround_time|prev_arr_delay|next_dep_delay|CHAIN_DELAY|
+-------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+--------+----------------+-------------------+--

#### These features are not relevant to delay prediction

In [23]:
df = df.dropDuplicates(["FL_DATE", "OP_CARRIER", "OP_CARRIER_FL_NUM", "CRS_DEP_TIME"])

In [24]:
row_count_cleaned = df.count()

col_count_cleaned = len(df.columns)

print(f"Shape: ({row_count_cleaned}, {col_count_cleaned})")

Shape: (6835982, 32)


> Shape before = <br> (7213446, 27) <br><br>
> Shape after Handling [nulls,duplication,add Chain_Delay feature] =(6835982, 32)


In [25]:
df.groupBy("CHAIN_DELAY").count().orderBy("CHAIN_DELAY").show()

+-----------+-------+
|CHAIN_DELAY|  count|
+-----------+-------+
|          0|6487650|
|          1| 348332|
+-----------+-------+



#### Easily We can notice the severe class imbalance
> Not Chain-Dealyed 6.487 Million VS 0.348 Million Chain-Dealyed

In [ ]:
df.limit(1).toPandas()

##### DROP ME LATER >> AFTER EDA, VISUALIZATION, FEATURE ENGINEERING

In [ ]:
# df = df.drop(
#     "ARR_TIME", "DEP_TIME", "WHEELS_OFF", "WHEELS_ON",
#     "ACTUAL_ELAPSED_TIME", "AIR_TIME", "TAXI_IN", "TAXI_OUT",
#     "ARR_DELAY",
#     "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
#     "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
#     "prev_crs_arr_minutes",
#     "CANCELLED", "DIVERTED"
# )

## Why Drop Each Group

**Post-flight actuals** — `ARR_TIME`, `DEP_TIME`, `WHEELS_OFF`, `WHEELS_ON`, `ACTUAL_ELAPSED_TIME`, `AIR_TIME`, `TAXI_IN`, `TAXI_OUT`, `ARR_DELAY`
These are recorded **after** the flight lands. In real prediction scenario you don't have them yet — keeping them would be data leakage.

---

**Delay causes** — `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY`
Already used to build `CHAIN_DELAY`. Their job is done, and they're also post-flight actuals anyway.

---

**`prev_crs_arr_minutes`** — Already used to compute `turnaround_time`. Redundant now.

---

**`CANCELLED`, `DIVERTED`** — You filtered them all to 0 during cleaning. Every single row is 0, so they carry **zero information** for the model.

In [ ]:
len(df.columns)

In [ ]:
df.limit(1).toPandas()